# Choice真实数据落库验收

本Notebook验证完整链路：

`Choice EmQuantAPI → Choice采集适配器 → SQLite → qianji OpenBB Provider → Excel/JSON证据`

运行前请确认：

1. `02_Choice_OpenBB插件真实验证.ipynb`已经正常运行；
2. VS Code选择的是安装了EmQuantAPI、`qianji-data-mini`和OpenBB的同一个Python内核；
3. `.env`中设置了`CHOICE_LOGIN_MODE=userinfo`；
4. 已运行`openbb-build`并重启内核；
5. Notebook不会打印或导出账号、密码及令牌。

## 1. 检查Python环境和项目目录

In [1]:
import os
import sys
from pathlib import Path

print("Python路径：", sys.executable)
print("Python版本：", sys.version.split()[0])
print("Conda环境：", os.getenv("CONDA_DEFAULT_ENV", "未检测到"))
print("Notebook当前目录：", Path.cwd().resolve())

Python路径： d:\minicoda3\envs\dm311\python.exe
Python版本： 3.11.14
Conda环境： dm311
Notebook当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks


In [2]:
# Notebook放在项目notebooks目录时无需修改。若单独存放，可填写项目根目录。
# 示例：PROJECT_ROOT_OVERRIDE = r"D:\\OneDrive\\桌面\\qianji_openbb_mini"
PROJECT_ROOT_OVERRIDE = ""

def find_project_root(start: Path) -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
        raise FileNotFoundError(f"指定的项目目录不正确：{candidate}")

    for candidate in (start, *start.parents):
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
    raise FileNotFoundError(
        "没有找到qianji_openbb_mini项目。请把Notebook放进项目的notebooks文件夹，"
        "或填写PROJECT_ROOT_OVERRIDE。"
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print("项目根目录：", PROJECT_ROOT)
print(".env是否存在：", (PROJECT_ROOT / ".env").exists())

项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
.env是否存在： True


## 2. 读取安全配置并检查组件

In [3]:
from dotenv import load_dotenv

ENV_PATH = PROJECT_ROOT / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(
        f"没有找到{ENV_PATH}。请复制.env.example为.env并填写Choice配置。"
    )

load_dotenv(ENV_PATH, override=True)
login_mode = os.getenv("CHOICE_LOGIN_MODE", "auto").strip().lower()
username_configured = bool(os.getenv("CHOICE_USERNAME", "").strip())
password_configured = bool(os.getenv("CHOICE_PASSWORD", ""))

print("Choice登录模式：", login_mode)
print("用户名已配置：", username_configured)
print("密码已配置：", password_configured)

if login_mode == "password" and not (username_configured and password_configured):
    raise RuntimeError("账密登录模式下必须同时配置CHOICE_USERNAME和CHOICE_PASSWORD。")
if login_mode not in {"auto", "password", "userinfo"}:
    raise RuntimeError("CHOICE_LOGIN_MODE只能填写auto、password或userinfo。")

Choice登录模式： userinfo
用户名已配置： True
密码已配置： True


In [4]:
from importlib.metadata import version

try:
    from EmQuantAPI import c as choice_sdk
    print("EmQuantAPI：导入成功")
except Exception as exc:
    raise RuntimeError("当前Notebook内核无法导入EmQuantAPI。") from exc

from qianji_data_mini import Database
from qianji_data_mini.ingest import ingest_daily
from openbb import obb

print("qianji-data-mini版本：", version("qianji-data-mini"))
print("OpenBB发现choice：", "choice" in obb.coverage.providers)
print("OpenBB发现qianji：", "qianji" in obb.coverage.providers)

if "qianji" not in obb.coverage.providers:
    raise RuntimeError(
        "OpenBB尚未发现qianji Provider。请重新安装根项目、执行openbb-build并重启内核。"
    )

EmQuantAPI：导入成功
qianji-data-mini版本： 0.3.2
OpenBB发现choice： True
OpenBB发现qianji： True


## 3. 设置验证范围和数据库

默认验证`.env`中`VALIDATION_SYMBOLS`的第一只证券；未填写日期时，验证截至昨天的最近45个自然日。

In [5]:
from datetime import date, timedelta

default_end = date.today() - timedelta(days=1)
default_start = default_end - timedelta(days=45)

SYMBOL = os.getenv("VALIDATION_SYMBOLS", "000001.SZ").split(",")[0].strip().upper()
START_DATE = os.getenv("VALIDATION_START_DATE", "").strip() or default_start.isoformat()
END_DATE = os.getenv("VALIDATION_END_DATE", "").strip() or default_end.isoformat()

database = Database()
DB_PATH = database.path

print("证券代码：", SYMBOL)
print("开始日期：", START_DATE)
print("结束日期：", END_DATE)
print("SQLite数据库：", DB_PATH)
print("数据库文件当前是否存在：", DB_PATH.exists())

证券代码： 000001.SZ
开始日期： 2026-07-16
结束日期： 2026-08-30
SQLite数据库： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\qianji_market.db
数据库文件当前是否存在： True


## 4. 第一次真实下载并写入SQLite

`stored_rows`表示本次执行写入或更新的记录数；真正的去重结果以下方数据库查询为准。

In [6]:
before_df = database.query_dataframe(
    symbol=SYMBOL, source="choice", start_date=START_DATE, end_date=END_DATE
)
count_before = len(before_df)
print("本次运行前数据库记录数：", count_before)

first_ingest = ingest_daily(
    source="choice",
    symbols=[SYMBOL],
    start_date=START_DATE,
    end_date=END_DATE,
    database_path=DB_PATH,
)

print(first_ingest.model_dump_json(indent=2))
if first_ingest.failed_symbols:
    raise RuntimeError(f"Choice落库失败：{first_ingest.failed_symbols}")
if first_ingest.received_rows == 0:
    raise RuntimeError("Choice没有返回数据，请检查代码、日期范围和数据权限。")

本次运行前数据库记录数： 0
[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:22]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:22]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:22]:connect server...

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:24]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:27]:updating ChoiceToHQ.xml from version 0 to 120

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:29]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:32]:DownLoad D:/EMQuantAPI_Python/python3/libs/windows/bjse_code_conversion.txt success.

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:33]:percentflag(for csd/css/cses) update success.

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:35]:heartbeatthread end.

{
  "source": "choice",
  "requested_symbols": [
    "000001.SZ"
  ],
  "received_rows": 32,
  "stored_rows": 32,
  "failed_symbols": {},
  "started_at": "2026-09-01T01:47:21.8364

## 5. 查询数据库并执行质量检查

In [7]:
import pandas as pd
from IPython.display import display

stored_df = database.query_dataframe(
    symbol=SYMBOL, source="choice", start_date=START_DATE, end_date=END_DATE
).sort_values("date").reset_index(drop=True)
count_after_first = len(stored_df)

required_fields = [
    "date", "open", "high", "low", "close", "volume",
    "amount", "previous_close", "change_percent", "source",
]
missing_by_field = {
    field: int(stored_df[field].isna().sum())
    for field in required_fields
}
duplicate_dates = int(stored_df.duplicated(subset=["date"]).sum())
ohlc_violations = int((
    (stored_df["low"] > stored_df[["open", "close"]].min(axis=1))
    | (stored_df["high"] < stored_df[["open", "close"]].max(axis=1))
    | (stored_df["low"] > stored_df["high"])
).sum())

print("第一次入库后记录数：", count_after_first)
print("重复日期：", duplicate_dates)
print("OHLC逻辑异常：", ohlc_violations)
print("数据库文件大小（字节）：", DB_PATH.stat().st_size)
display(database.source_status())
display(stored_df.head(10))
display(pd.DataFrame({"字段": missing_by_field.keys(), "缺失数": missing_by_field.values()}))

第一次入库后记录数： 32
重复日期： 0
OHLC逻辑异常： 0
数据库文件大小（字节）： 45056


,source,rows,symbols,first_date,last_date,last_ingested_at
0,choice,32,1,2026-07-16,2026-08-28,2026-09-01T01:47:35.896279+00:00
1,mock,42,2,2026-07-31,2026-08-28,2026-08-31T03:46:28.634076+00:00


,source,symbol,adjustment,open,high,low,close,volume,amount,previous_close,change_percent,currency,timezone,volume_unit,amount_unit,ingested_at,date
0,choice,000001.SZ,unadjusted,10.85,10.93,10.72,10.77,80076623.0,8.644359e+08,10.84,-0.6458,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-16
1,choice,000001.SZ,unadjusted,10.75,10.88,10.72,10.78,107549901.0,1.163189e+09,10.77,0.0929,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-17
2,choice,000001.SZ,unadjusted,10.75,11.00,10.74,10.98,156730393.0,1.713460e+09,10.78,1.8553,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-20
3,choice,000001.SZ,unadjusted,10.99,11.13,10.83,10.84,175511288.0,1.925299e+09,10.98,-1.2750,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-21
4,choice,000001.SZ,unadjusted,10.81,10.98,10.77,10.98,102948394.0,1.120151e+09,10.84,1.2915,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-22
5,choice,000001.SZ,unadjusted,10.92,11.12,10.90,11.08,109574268.0,1.210838e+09,10.98,0.9107,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-23
6,choice,000001.SZ,unadjusted,11.09,11.18,11.09,11.10,114093292.0,1.269361e+09,11.08,0.1805,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-24
7,choice,000001.SZ,unadjusted,11.11,11.16,11.04,11.11,95715556.0,1.062796e+09,11.10,0.0901,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-27
8,choice,000001.SZ,unadjusted,11.10,11.21,11.09,11.20,106101129.0,1.185515e+09,11.11,0.8101,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-28
9,choice,000001.SZ,unadjusted,11.19,11.36,11.18,11.28,151105407.0,1.705864e+09,11.20,0.7143,CNY,Asia/Shanghai,share,CNY,2026-09-01T01:47:35.895732+00:00,2026-07-29


,字段,缺失数
0,date,0
1,open,0
2,high,0
3,low,0
4,close,0
5,volume,0
6,amount,0
7,previous_close,0
8,change_percent,0
9,source,0


## 6. 第二次重复入库，验证幂等性

相同来源、证券、日期和复权口径组成唯一键。重复执行后数据库记录数应保持不变。

In [8]:
second_ingest = ingest_daily(
    source="choice",
    symbols=[SYMBOL],
    start_date=START_DATE,
    end_date=END_DATE,
    database_path=DB_PATH,
)

if second_ingest.failed_symbols:
    raise RuntimeError(f"第二次Choice落库失败：{second_ingest.failed_symbols}")

stored_after_second_df = database.query_dataframe(
    symbol=SYMBOL, source="choice", start_date=START_DATE, end_date=END_DATE
)
count_after_second = len(stored_after_second_df)
idempotent = count_after_first == count_after_second

print("第一次入库后记录数：", count_after_first)
print("第二次入库后记录数：", count_after_second)
print("幂等写入是否通过：", idempotent)

if not idempotent:
    raise RuntimeError("重复入库后记录数发生变化，幂等校验失败。")

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:54]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:54]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:54]:connect server...

[EmQuantAPI Python] [Em_Info][2026-08-31 18:47:57]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-08-31 18:48:00]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-08-31 18:48:04]:heartbeatthread end.

第一次入库后记录数： 32
第二次入库后记录数： 32
幂等写入是否通过： True


## 7. 使用qianji Provider从SQLite读取

In [9]:
qianji_result = obb.equity.price.historical(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    provider="qianji",
    source="choice",
)

openbb_rows = [
    item.model_dump(mode="json") if hasattr(item, "model_dump") else dict(item)
    for item in qianji_result.results
]
openbb_df = pd.DataFrame(openbb_rows).sort_values("date").reset_index(drop=True)

print("OpenBB provider：", qianji_result.provider)
print("原始数据源：", openbb_df["source"].unique().tolist())
print("OpenBB读取行数：", len(openbb_df))
display(openbb_df.head(10))

OpenBB provider： qianji
原始数据源： ['choice']
OpenBB读取行数： 32


,date,open,high,low,close,volume,vwap,source,amount,previous_close,change_percent,currency,timezone,volume_unit,amount_unit,symbol,adjustment,ingested_at
0,2026-07-16,10.85,10.93,10.72,10.77,80076623.0,None,choice,8.644359e+08,10.84,-0.006458,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00
1,2026-07-17,10.75,10.88,10.72,10.78,107549901.0,None,choice,1.163189e+09,10.77,0.000929,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00
2,2026-07-20,10.75,11.00,10.74,10.98,156730393.0,None,choice,1.713460e+09,10.78,0.018553,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00
3,2026-07-21,10.99,11.13,10.83,10.84,175511288.0,None,choice,1.925299e+09,10.98,-0.012750,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00
4,2026-07-22,10.81,10.98,10.77,10.98,102948394.0,None,choice,1.120151e+09,10.84,0.012915,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00
5,2026-07-23,10.92,11.12,10.90,11.08,109574268.0,None,choice,1.210838e+09,10.98,0.009107,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00
6,2026-07-24,11.09,11.18,11.09,11.10,114093292.0,None,choice,1.269361e+09,11.08,0.001805,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00
7,2026-07-27,11.11,11.16,11.04,11.11,95715556.0,None,choice,1.062796e+09,11.10,0.000901,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00
8,2026-07-28,11.10,11.21,11.09,11.20,106101129.0,None,choice,1.185515e+09,11.11,0.008101,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00
9,2026-07-29,11.19,11.36,11.18,11.28,151105407.0,None,choice,1.705864e+09,11.20,0.007143,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T01:48:04.632358+00:00


## 8. 比对SQLite与OpenBB回读结果

SQLite保存Choice原始百分点；OpenBB输出标准化小数。例如数据库`0.6009`对应OpenBB的`0.006009`。

In [10]:
import numpy as np

stored_compare = stored_df.copy()
stored_compare["date"] = stored_compare["date"].astype(str)
stored_compare["change_percent"] = stored_compare["change_percent"] / 100
openbb_compare = openbb_df.copy()
openbb_compare["date"] = openbb_compare["date"].astype(str)

comparison_fields = [
    "open", "high", "low", "close", "volume", "amount",
    "previous_close", "change_percent",
]
comparison = stored_compare[["date", *comparison_fields]].merge(
    openbb_compare[["date", *comparison_fields]],
    on="date",
    how="outer",
    suffixes=("_sqlite", "_openbb"),
    indicator=True,
)

comparison_mismatches = {}
for field in comparison_fields:
    left = pd.to_numeric(comparison[f"{field}_sqlite"], errors="coerce")
    right = pd.to_numeric(comparison[f"{field}_openbb"], errors="coerce")
    equal = np.isclose(left, right, rtol=1e-9, atol=1e-9, equal_nan=True)
    comparison_mismatches[field] = int((~equal).sum())

date_join_mismatches = int((comparison["_merge"] != "both").sum())
total_value_mismatches = sum(comparison_mismatches.values())

print("日期集合差异：", date_join_mismatches)
print("字段值差异合计：", total_value_mismatches)
display(pd.DataFrame({"字段": comparison_mismatches.keys(), "差异数": comparison_mismatches.values()}))

if date_join_mismatches or total_value_mismatches:
    raise RuntimeError("SQLite与OpenBB回读结果存在差异。")

日期集合差异： 0
字段值差异合计： 0


,字段,差异数
0,open,0
1,high,0
2,low,0
3,close,0
4,volume,0
5,amount,0
6,previous_close,0
7,change_percent,0


## 9. 查看本次采集运行记录

In [11]:
with database.connect() as connection:
    ingestion_runs_df = pd.read_sql_query(
        """
        SELECT run_id, source, requested_symbols, start_date, end_date,
               received_rows, stored_rows, failed_symbols, started_at, finished_at
        FROM ingestion_run
        WHERE source = 'choice'
        ORDER BY run_id DESC
        LIMIT 10
        """,
        connection,
    )

display(ingestion_runs_df.head())

,run_id,source,requested_symbols,start_date,end_date,received_rows,stored_rows,failed_symbols,started_at,finished_at
0,4,choice,"[""000001.SZ""]",2026-07-16,2026-08-30,32,32,{},2026-09-01T01:47:53.742264+00:00,2026-09-01T01:48:04.646221+00:00
1,3,choice,"[""000001.SZ""]",2026-07-16,2026-08-30,32,32,{},2026-09-01T01:47:21.836428+00:00,2026-09-01T01:47:35.913571+00:00


## 10. 汇总验收结论

In [12]:
key_missing_total = sum(missing_by_field.values())
evidence = {
    "status": "PASS",
    "source": "choice",
    "symbol": SYMBOL,
    "requested_start_date": START_DATE,
    "requested_end_date": END_DATE,
    "database_path": str(DB_PATH),
    "database_exists": DB_PATH.exists(),
    "database_size_bytes": DB_PATH.stat().st_size,
    "rows_before_run": count_before,
    "first_received_rows": first_ingest.received_rows,
    "rows_after_first_ingest": count_after_first,
    "rows_after_second_ingest": count_after_second,
    "idempotent": idempotent,
    "duplicate_date_count": duplicate_dates,
    "ohlc_violation_count": ohlc_violations,
    "key_missing_total": key_missing_total,
    "missing_by_field": missing_by_field,
    "openbb_provider": str(qianji_result.provider),
    "openbb_source": "choice",
    "openbb_row_count": len(openbb_df),
    "date_join_mismatches": date_join_mismatches,
    "value_mismatches": comparison_mismatches,
    "credentials_included": False,
}

checks = [
    first_ingest.received_rows > 0,
    count_after_first > 0,
    idempotent,
    duplicate_dates == 0,
    ohlc_violations == 0,
    key_missing_total == 0,
    str(qianji_result.provider) == "qianji",
    len(openbb_df) == count_after_first,
    date_join_mismatches == 0,
    total_value_mismatches == 0,
]
if not all(checks):
    evidence["status"] = "FAIL"
    raise RuntimeError(f"验收未通过：{evidence}")

display(pd.DataFrame([{
    key: value for key, value in evidence.items()
    if not isinstance(value, dict)
}]))
print("Choice真实数据落库验收：PASS")

,status,source,symbol,requested_start_date,requested_end_date,database_path,database_exists,database_size_bytes,rows_before_run,first_received_rows,...,rows_after_second_ingest,idempotent,duplicate_date_count,ohlc_violation_count,key_missing_total,openbb_provider,openbb_source,openbb_row_count,date_join_mismatches,credentials_included
0,PASS,choice,000001.SZ,2026-07-16,2026-08-30,D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\...,True,45056,0,32,...,32,True,0,0,0,qianji,choice,32,0,False


Choice真实数据落库验收：PASS


## 11. 导出Excel和JSON硬证据

导出内容包括验收摘要、SQLite真实数据、OpenBB回读数据、两次采集记录和字段比对，不包含账号密码。

In [13]:
import json
from datetime import datetime

OUTPUT_DIR = PROJECT_ROOT / "validation_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
safe_symbol = SYMBOL.replace(".", "_")
excel_path = OUTPUT_DIR / f"Choice真实落库验收_{safe_symbol}_{timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice真实落库验收_{safe_symbol}_{timestamp}.json"

summary_for_excel = {
    key: json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value
    for key, value in evidence.items()
}
missing_df = pd.DataFrame({"字段": missing_by_field.keys(), "缺失数": missing_by_field.values()})
mismatch_df = pd.DataFrame({"字段": comparison_mismatches.keys(), "差异数": comparison_mismatches.values()})

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    pd.DataFrame([summary_for_excel]).to_excel(writer, sheet_name="验收摘要", index=False)
    stored_df.to_excel(writer, sheet_name="SQLite真实数据", index=False)
    openbb_df.to_excel(writer, sheet_name="OpenBB回读", index=False)
    ingestion_runs_df.to_excel(writer, sheet_name="采集运行记录", index=False)
    missing_df.to_excel(writer, sheet_name="字段缺失统计", index=False)
    mismatch_df.to_excel(writer, sheet_name="数据比对", index=False)

json_payload = {
    "evidence": evidence,
    "sqlite_results": stored_df.to_dict(orient="records"),
    "openbb_results": openbb_rows,
    "ingestion_runs": ingestion_runs_df.to_dict(orient="records"),
}
json_path.write_text(
    json.dumps(json_payload, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("数据库文件：", DB_PATH)
print("Excel验收证据：", excel_path)
print("JSON验收证据：", json_path)

数据库文件： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\qianji_market.db
Excel验收证据： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice真实落库验收_000001_SZ_20260831_184828.xlsx
JSON验收证据： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice真实落库验收_000001_SZ_20260831_184828.json


## 验收完成标准

最后显示`Choice真实数据落库验收：PASS`，并成功生成SQLite、Excel和JSON时，可以证明：

- Choice真实数据已经写入本地SQLite数据库；
- 重复执行不会产生重复记录；
- `provider="qianji", source="choice"`能够从数据库回读；
- SQLite与OpenBB回读的日期、价格、量额及涨跌幅口径一致。

> 注意：这里验证的是轻量MVP的本地SQLite落库，不等同于公司正式生产数据库部署。